<a href="https://colab.research.google.com/github/NidhalSoumri/multiformat-summarization-rouge/blob/main/Summary.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q langchain langchain-google-genai langchain-community pypdf rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
print("Clé API configurée ✅")

Clé API configurée ✅


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

chat_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3  # Un peu de créativité pour la génération de résumés
)

print("Modèle prêt ✅")

Modèle prêt ✅


In [ ]:
# Texte source — extrait Wikipedia sur Docker
texte_source = """
Docker est une plateforme permettant de lancer certaines applications dans des conteneurs logiciels lancée en 2013.
Selon une étude menée par Cloud Foundry, Docker est l'outil de containerisation le plus utilisé en entreprise.

Docker est un logiciel libre qui automatise le déploiement d'applications dans des conteneurs logiciels.
Selon le cabinet d'analyse industrielle 451 Research, Docker est un outil qui peut empaqueter une application
et ses dépendances dans un conteneur isolé, qui pourra être exécuté sur n'importe quel serveur.
Ceci permet d'étendre la flexibilité et la portabilité d'exécution d'une application, que ce soit sur la machine
locale, un cloud privé ou public, une machine nue, etc.

Docker met en œuvre des fonctionnalités de haut niveau qui sont :
- L'amélioration de la conception modulaire, par sa capacité à empaqueter une application et toutes ses dépendances dans un conteneur,
- Les diffusions versionnées par une gestion des images sous la forme de couches,
- Le partage du système de fichiers pour économiser l'espace,
- La gestion des conteneurs et des images Docker.

Docker utilise le noyau Linux et ses fonctionnalités, comme les Cgroups et les espaces de noms, pour permettre
l'exécution de conteneurs indépendants au sein d'un même système d'exploitation, évitant ainsi la charge
de démarrer une machine virtuelle entière. Le noyau Linux dans les espaces de noms isole la vue qu'a
une application de son environnement d'exploitation, y compris les arborescences de processus, le réseau,
les identifiants d'utilisateur et les systèmes de fichiers montés.

Depuis la version 0.9, Docker inclut une bibliothèque appelée libcontainer pour utiliser directement
les facilités de virtualisation offertes par le noyau Linux, en plus d'utiliser des interfaces
d'abstraction via libvirt, LXC et systemd-nspawn.

Selon une enquête menée en 2018 par Datadog auprès de plus de 10 000 entreprises clientes, près d'un quart
des entreprises utilisent Docker. Au 23 mai 2016, 8 grands projets sont les principaux développeurs des
plus de 540 contributeurs en open source au projet Docker : Docker, Red Hat, Google, Cisco, IBM, Microsoft,
Huawei et Amadeus IT Group.
"""

# Affichage des statistiques
print(f"📄 Texte chargé")
print(f"   Longueur : {len(texte_source)} caractères")
print(f"   Nombre de mots : {len(texte_source.split())} mots")
print(f"   Premier aperçu :\n")
print(texte_source[:300] + "...")

📄 Texte chargé
   Longueur : 2204 caractères
   Nombre de mots : 334 mots
   Premier aperçu :


Docker est une plateforme permettant de lancer certaines applications dans des conteneurs logiciels lancée en 2013. 
Selon une étude menée par Cloud Foundry, Docker est l'outil de containerisation le plus utilisé en entreprise.

Docker est un logiciel libre qui automatise le déploiement d'applicati...


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prompt baseline ultra simple
baseline_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant qui résume des textes de manière concise et fidèle."),
    ("human", "Résume le texte suivant en 3-4 phrases :\n\n{texte}")
])

# Chaîne simple
baseline_chain = baseline_prompt | chat_model | StrOutputParser()

# Génération du résumé
print("⏳ Génération du résumé baseline...\n")
resume_baseline = baseline_chain.invoke({"texte": texte_source})

print("📝 Résumé baseline :\n")
print(resume_baseline)
print(f"\n📊 Longueur du résumé : {len(resume_baseline)} caractères, {len(resume_baseline.split())} mots")

⏳ Génération du résumé baseline...

📝 Résumé baseline :

Docker est une plateforme logicielle libre lancée en 2013 qui automatise le déploiement d'applications dans des conteneurs. Il permet d'empaqueter une application et toutes ses dépendances dans un conteneur isolé, assurant ainsi une exécution flexible et portable sur n'importe quel serveur.

Cette technologie utilise les fonctionnalités du noyau Linux pour exécuter des conteneurs indépendants sans la charge d'une machine virtuelle. Docker améliore la conception modulaire et la gestion des applications grâce à des fonctionnalités comme les diffusions versionnées et le partage de système de fichiers. C'est l'outil de conteneurisation le plus utilisé en entreprise et un projet open source majeur soutenu par de grandes entreprises.

📊 Longueur du résumé : 737 caractères, 105 mots


In [ ]:
# ============================================
# Stratégie 1 : ZERO-SHOT
# ============================================
prompt_zero_shot = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant qui résume des textes de manière concise et fidèle."),
    ("human", "Résume le texte suivant en 3-4 phrases :\n\n{texte}")
])

# ============================================
# Stratégie 2 : FEW-SHOT
# Le modèle voit un exemple avant de faire la vraie tâche
# ============================================
prompt_few_shot = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant qui résume des textes de manière concise et fidèle. Tu suis le format des exemples fournis."),
    ("human", """Voici un exemple :

TEXTE :
Python est un langage de programmation interprété, multi-paradigme et multiplateformes. Il favorise la programmation impérative structurée, fonctionnelle et orientée objet. Il est doté d'un typage dynamique fort, d'une gestion automatique de la mémoire par ramasse-miettes et d'un système de gestion d'exceptions. Le langage Python est placé sous une licence libre proche de la licence BSD.

RÉSUMÉ :
Python est un langage de programmation interprété, multi-paradigme et multiplateforme, supportant la programmation impérative, fonctionnelle et orientée objet. Il possède un typage dynamique fort, une gestion automatique de la mémoire et un système d'exceptions. C'est un langage libre sous licence proche de BSD.

Maintenant, résume ce nouveau texte en suivant le même format et style :

TEXTE :
{texte}

RÉSUMÉ :""")
])

# ============================================
# Stratégie 3 : CHAIN-OF-THOUGHT
# On force le modèle à raisonner étape par étape
# ============================================
prompt_cot = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant qui résume des textes de manière structurée et fidèle, en raisonnant étape par étape."),
    ("human", """Pour résumer ce texte, suis ces étapes dans ta tête :

1. Identifie les 3 à 5 idées principales du texte
2. Élimine les détails secondaires, exemples et chiffres non essentiels
3. Reformule ces idées de manière cohérente en 3-4 phrases

Donne UNIQUEMENT le résumé final (sans afficher tes étapes de réflexion).

TEXTE :
{texte}

RÉSUMÉ FINAL :""")
])

print("✅ Les 3 prompts sont définis")

✅ Les 3 prompts sont définis


In [ ]:
# Construction des 3 chaînes
chain_zero_shot = prompt_zero_shot | chat_model | StrOutputParser()
chain_few_shot = prompt_few_shot | chat_model | StrOutputParser()
chain_cot = prompt_cot | chat_model | StrOutputParser()

# Génération des 3 résumés
print("⏳ Génération des 3 résumés...\n")

resume_zero_shot = chain_zero_shot.invoke({"texte": texte_source})
print("=" * 60)
print("📝 STRATÉGIE 1 : ZERO-SHOT")
print("=" * 60)
print(resume_zero_shot)
print(f"\n📊 {len(resume_zero_shot)} caractères, {len(resume_zero_shot.split())} mots\n")

resume_few_shot = chain_few_shot.invoke({"texte": texte_source})
print("=" * 60)
print("📝 STRATÉGIE 2 : FEW-SHOT")
print("=" * 60)
print(resume_few_shot)
print(f"\n📊 {len(resume_few_shot)} caractères, {len(resume_few_shot.split())} mots\n")

resume_cot = chain_cot.invoke({"texte": texte_source})
print("=" * 60)
print("📝 STRATÉGIE 3 : CHAIN-OF-THOUGHT")
print("=" * 60)
print(resume_cot)
print(f"\n📊 {len(resume_cot)} caractères, {len(resume_cot.split())} mots")

⏳ Génération des 3 résumés...

📝 STRATÉGIE 1 : ZERO-SHOT
Docker est une plateforme logicielle libre, lancée en 2013, qui automatise le déploiement d'applications dans des conteneurs. Il permet d'empaqueter une application et ses dépendances dans un environnement isolé, offrant ainsi une grande flexibilité et portabilité sur divers serveurs ou clouds. Largement adopté en entreprise, Docker utilise les fonctionnalités du noyau Linux pour exécuter ces conteneurs indépendamment, évitant la charge des machines virtuelles. Le projet est open source et bénéficie des contributions de nombreuses entreprises majeures.

📊 558 caractères, 76 mots

📝 STRATÉGIE 2 : FEW-SHOT
RÉSUMÉ :
Docker est une plateforme libre lancée en 2013, qui automatise le déploiement d'applications dans des conteneurs logiciels. C'est l'outil de conteneurisation le plus utilisé en entreprise. Il empaquette une application et ses dépendances dans un conteneur isolé, offrant flexibilité et portabilité. Ses fonctionnalités inc

In [ ]:
# Résumé de référence (idéalement écrit par un humain)
# C'est ce que TOI tu considères être un "bon" résumé
resume_reference = """Docker est une plateforme logicielle libre lancée en 2013 qui automatise le déploiement d'applications
dans des conteneurs isolés. Elle permet d'empaqueter une application et ses dépendances pour les exécuter sur n'importe
quel serveur, offrant flexibilité et portabilité. Docker utilise le noyau Linux et ses fonctionnalités comme les Cgroups
et les espaces de noms, évitant la charge d'une machine virtuelle. C'est l'outil de conteneurisation le plus utilisé
en entreprise selon plusieurs études."""

print("📋 Résumé de référence :")
print(resume_reference)
print(f"\n📊 {len(resume_reference)} caractères, {len(resume_reference.split())} mots")

📋 Résumé de référence :
Docker est une plateforme logicielle libre lancée en 2013 qui automatise le déploiement d'applications 
dans des conteneurs isolés. Elle permet d'empaqueter une application et ses dépendances pour les exécuter sur n'importe 
quel serveur, offrant flexibilité et portabilité. Docker utilise le noyau Linux et ses fonctionnalités comme les Cgroups 
et les espaces de noms, évitant la charge d'une machine virtuelle. C'est l'outil de conteneurisation le plus utilisé 
en entreprise selon plusieurs études.

📊 502 caractères, 71 mots


In [ ]:
from rouge_score import rouge_scorer

# Initialisation du scorer avec ROUGE-1, ROUGE-2 et ROUGE-L
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Calcul des scores pour chaque stratégie
print("🔬 Calcul des scores ROUGE pour les 3 stratégies\n")
print("=" * 75)

resultats = {}

for nom, resume in [
    ("Zero-shot", resume_zero_shot),
    ("Few-shot", resume_few_shot),
    ("Chain-of-Thought", resume_cot)
]:
    scores = scorer.score(resume_reference, resume)
    resultats[nom] = scores

    print(f"\n📝 Stratégie : {nom}")
    print("-" * 40)
    for metric_name, score in scores.items():
        print(f"   {metric_name:8} | Precision: {score.precision:.3f} | Recall: {score.recall:.3f} | F1: {score.fmeasure:.3f}")

print("\n" + "=" * 75)
print("🏆 Comparaison des F1-scores (ROUGE-L) :")
print("=" * 75)
for nom, scores in resultats.items():
    f1_l = scores['rougeL'].fmeasure
    print(f"   {nom:20} → ROUGE-L F1 = {f1_l:.3f}")

🔬 Calcul des scores ROUGE pour les 3 stratégies


📝 Stratégie : Zero-shot
----------------------------------------
   rouge1   | Precision: 0.686 | Recall: 0.711 | F1: 0.698
   rouge2   | Precision: 0.435 | Recall: 0.451 | F1: 0.443
   rougeL   | Precision: 0.523 | Recall: 0.542 | F1: 0.533

📝 Stratégie : Few-shot
----------------------------------------
   rouge1   | Precision: 0.559 | Recall: 0.795 | F1: 0.657
   rouge2   | Precision: 0.393 | Recall: 0.561 | F1: 0.462
   rougeL   | Precision: 0.347 | Recall: 0.494 | F1: 0.408

📝 Stratégie : Chain-of-Thought
----------------------------------------
   rouge1   | Precision: 0.584 | Recall: 0.627 | F1: 0.605
   rouge2   | Precision: 0.318 | Recall: 0.341 | F1: 0.329
   rougeL   | Precision: 0.427 | Recall: 0.458 | F1: 0.442

🏆 Comparaison des F1-scores (ROUGE-L) :
   Zero-shot            → ROUGE-L F1 = 0.533
   Few-shot             → ROUGE-L F1 = 0.408
   Chain-of-Thought     → ROUGE-L F1 = 0.442


In [ ]:
%pip install -q rouge-score

In [ ]:
import gradio as gr
from rouge_score import rouge_scorer

# Initialisation du scorer (au cas où la cellule précédente n'a pas tourné)
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def comparer_strategies(texte, reference):
    """Génère 3 résumés avec 3 stratégies et calcule les scores ROUGE"""

    if not texte.strip():
        return "⚠️ Merci de fournir un texte à résumer", "", "", ""

    # Génération des 3 résumés
    r_zs = chain_zero_shot.invoke({"texte": texte})
    r_fs = chain_few_shot.invoke({"texte": texte})
    r_cot = chain_cot.invoke({"texte": texte})

    # Si pas de référence fournie, on retourne juste les résumés
    if not reference.strip():
        message_rouge = "ℹ️ Aucune référence fournie — les scores ROUGE n'ont pas été calculés."
        return r_zs, r_fs, r_cot, message_rouge

    # Calcul des scores ROUGE
    scores_zs = scorer.score(reference, r_zs)
    scores_fs = scorer.score(reference, r_fs)
    scores_cot = scorer.score(reference, r_cot)

    # Construction du tableau de comparaison
    tableau = f"""
| Stratégie         | ROUGE-1 F1 | ROUGE-2 F1 | ROUGE-L F1 |
|-------------------|------------|------------|------------|
| Zero-shot         | {scores_zs['rouge1'].fmeasure:.3f}      | {scores_zs['rouge2'].fmeasure:.3f}      | {scores_zs['rougeL'].fmeasure:.3f}      |
| Few-shot          | {scores_fs['rouge1'].fmeasure:.3f}      | {scores_fs['rouge2'].fmeasure:.3f}      | {scores_fs['rougeL'].fmeasure:.3f}      |
| Chain-of-Thought  | {scores_cot['rouge1'].fmeasure:.3f}      | {scores_cot['rouge2'].fmeasure:.3f}      | {scores_cot['rougeL'].fmeasure:.3f}      |
"""

    # Identification du gagnant sur ROUGE-L
    scores_l = {
        "Zero-shot": scores_zs['rougeL'].fmeasure,
        "Few-shot": scores_fs['rougeL'].fmeasure,
        "Chain-of-Thought": scores_cot['rougeL'].fmeasure
    }
    gagnant = max(scores_l, key=scores_l.get)

    rapport = f"{tableau}\n\n🏆 **Meilleure stratégie sur ROUGE-L F1 : {gagnant}** ({scores_l[gagnant]:.3f})"

    return r_zs, r_fs, r_cot, rapport


# Construction de l'interface
demo = gr.Interface(
    fn=comparer_strategies,
    inputs=[
        gr.Textbox(
            label="📄 Texte à résumer",
            placeholder="Colle ici un article, un extrait, un paragraphe...",
            lines=10,
            value=texte_source  # pré-rempli avec le texte Docker
        ),
        gr.Textbox(
            label="📋 Résumé de référence (optionnel — pour les scores ROUGE)",
            placeholder="Si tu fournis un résumé idéal, ROUGE comparera les 3 stratégies.",
            lines=5,
            value=resume_reference  # pré-rempli avec ta référence
        )
    ],
    outputs=[
        gr.Textbox(label="📝 Zero-shot", lines=6),
        gr.Textbox(label="📝 Few-shot", lines=6),
        gr.Textbox(label="📝 Chain-of-Thought", lines=6),
        gr.Markdown(label="📊 Comparaison ROUGE")
    ],
    title="📚 Outil de Résumé Multiformat — Comparaison de 3 Stratégies",
    description="""Compare 3 stratégies de prompt engineering (zero-shot, few-shot, chain-of-thought)
    pour la génération de résumés, avec évaluation objective via ROUGE-1, ROUGE-2 et ROUGE-L.""",
    theme=gr.themes.Soft(),
    flagging_mode="never"
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed9f3be9987d80fc1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
